# Application on Dataset - ConceptNet Context

In [ ]:
# ==============================================================================
# CELL 1: SETUP, IMPORTS, AND DATA LOADING
# ==============================================================================
!pip install -q -U google-generativeai pandas scikit-learn tqdm seaborn matplotlib

import pandas as pd
import google.generativeai as genai
from google.colab import userdata
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import re
import time
import random
from google.colab import drive

# --- API and Drive Configuration ---
try:
    # IMPORTANT: Replace with your key or use Colab Secrets
    genai.configure(api_key='')
    print("✅ API do Gemini configurada.")
except Exception as e:
    print(f"⚠️ Chave de API não encontrada ou inválida. Erro: {e}")

try:
    drive.mount('/content/drive')
    print("✅ Google Drive montado.")
except:
    print("ℹ️ Google Drive já montado ou erro ao montar.")

# --- Load Initial Prompts File ---
PROMPTS_FILE_PATH = '/content/drive/My Drive/.../aae_test_prompts.csv'
try:
    df = pd.read_csv(PROMPTS_FILE_PATH)
    if 'classification' in df.columns and 'ground_truth_stance' not in df.columns:
        df.rename(columns={'classification': 'ground_truth_stance'}, inplace=True)
    print(f"✅ Arquivo '{PROMPTS_FILE_PATH}' carregado. Total de {len(df)} linhas.")
    display(df.head())
except FileNotFoundError:
    print(f"❌ ERRO: Arquivo '{PROMPTS_FILE_PATH}' não encontrado.")
    df = None

In [ ]:
df.info()

In [ ]:
# ==============================================================================
# CELL 2: STEP 1 - SENTIMENT ANALYSIS (With Error Handling)
# ==============================================================================

def _call_api_with_retry(prompt, model, expected_responses):
    """
    A robust function to call the Gemini API with retries and exponential backoff.
    """
    max_retries = 5
    for attempt in range(max_retries):
        try:
            # Add a small, random delay to space out requests
            time.sleep(random.uniform(0.5, 1.5))
            response = model.generate_content(prompt)
            answer = response.text.strip().upper()

            for resp in expected_responses:
                if resp in answer:
                    return resp
            return "N/A" # Return N/A if no expected response is found

        except Exception as e:
            if "timed out" in str(e).lower() or "connection" in str(e).lower():
                wait_time = (2 ** attempt) + random.random() # Exponential backoff
                print(f"⚠️ Timeout/Connection error. Tentativa {attempt + 1}/{max_retries}. Esperando {wait_time:.2f}s...")
                time.sleep(wait_time)
            else:
                print(f"❌ Erro não relacionado a timeout: {str(e)}")
                return "ERROR"

    print(f"🚨 Todas as {max_retries} tentativas falharam para o prompt.")
    return "ERROR"


def get_sentiment(text, model):
    """Classifies the sentiment of a text using the robust API call function."""
    prompt = f"Analyze the sentiment of the following text. Respond ONLY with the word POSITIVE, NEGATIVE, or NEUTRAL.\n\nText: \"{text}\"\n\nSentiment:"
    return _call_api_with_retry(prompt, model, ["POSITIVE", "NEGATIVE", "NEUTRAL"])

if df is not None:
    print("\n--- Iniciando Etapa 1: Análise de Sentimento ---")
    model = genai.GenerativeModel('gemini-1.5-flash')

    # Run sentiment analysis for arguments
    with ThreadPoolExecutor(max_workers=5) as executor: # Reduced workers to be safer
        futures = {executor.submit(get_sentiment, text, model): i for i, text in enumerate(df['argument'])}
        arg_sentiments = [None] * len(df)
        for future in tqdm(as_completed(futures), total=len(df), desc="Analisando sentimento dos argumentos"):
            arg_sentiments[futures[future]] = future.result()
    df['argument_sentiment'] = arg_sentiments

    # Run sentiment analysis for topics
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(get_sentiment, text, model): i for i, text in enumerate(df['topic'])}
        topic_sentiments = [None] * len(df)
        for future in tqdm(as_completed(futures), total=len(df), desc="Analisando sentimento dos tópicos"):
            topic_sentiments[futures[future]] = future.result()
    df['topic_sentiment'] = topic_sentiments

    # Save the intermediate result
    output_sentiment_file = '/content/drive/My Drive/.../aae_with_sentiment.csv'
    df.to_csv(output_sentiment_file, index=False, encoding='utf-8')
    print(f"\n✅ Análise de sentimento concluída. Resultados salvos em '{output_sentiment_file}'")
    display(df.head())

In [ ]:
# ==============================================================================
# CELL 3: STEPS 2-7 - RUNNING EXPERIMENTS SEQUENTIALLY (With Error Handling)
# ==============================================================================

# Load the file with sentiment scores
CURRENT_FILE_PATH = '/content/drive/My Drive/.../aae_with_sentiment.csv'
try:
    df_exp = pd.read_csv(CURRENT_FILE_PATH)
    print(f"✅ Arquivo '{CURRENT_FILE_PATH}' carregado para iniciar os experimentos.")
except FileNotFoundError:
    print(f"❌ ERRO: Arquivo intermediário '{CURRENT_FILE_PATH}' não encontrado.")
    df_exp = None

if df_exp is not None:
    model = genai.GenerativeModel('gemini-1.5-flash')

    # --- Define a function for each experiment ---
    def process_exp1_no_context(row):
        prompt = f"Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.\n\nQuestion: The argument \"{row['argument']}\" SUPPORT or ATTACK the belief \"{row['topic']}\"?"
        return _call_api_with_retry(prompt, model, ["ATTACK", "SUPPORT"])

    def process_exp2_with_kg(row):
        return _call_api_with_retry(row['final_prompt'], model, ["ATTACK", "SUPPORT"])

    def process_exp3_with_sentiment(row):
        prompt = (f"Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' and sentiment analysis. Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.\n\n"
                  f"Sentiment Analysis:\nArgument has a {row['argument_sentiment']} sentiment\nTopic has a {row['topic_sentiment']} sentiment\n\n"
                  f"Knowledge Graph Context: {row['knowledge_context']}\n\n"
                  f"Question: The argument \"{row['argument']}\" SUPPORT or ATTACK the belief \"{row['topic']}\"?")
        return _call_api_with_retry(prompt, model, ["ATTACK", "SUPPORT"])

    def process_exp4_cot(row):
        prompt = (f"{row['final_prompt']}\n\nReason in the following way:\n1. Analyze the core assertion of the argument and the topic.\n"
                  f"2. Does the argument provide evidence, data, or reasoning that directly confirms or contradicts the topic?\n"
                  f"3. Based on this relationship, conclude if the argument supports or attacks the topic.\nConclusion:")
        return _call_api_with_retry(prompt, model, ["ATTACK", "SUPPORT"])

    def process_exp5_few_shot(row):
        example_1 = ("Argument: \"IWhat we acquired from team work is not only how to achieve the same goal with others but more importantly, how to get along with other\"\n"
                     "Belief: \"through cooperation, children can learn about interpersonal skills which are significant in the future life of all students.\"\nAnswer: SUPPORT")
        example_2 = ("Argument: \"competition makes the society more effective\"\n"
                     "Belief: \"we should attach more importance to cooperation during primary education\"\nAnswer: ATTACK")
        header = f"Here are some examples:\n\n[EXAMPLE 1]\n{example_1}\n\n[EXAMPLE 2]\n{example_2}\n\n---\n\nNow, complete the following task:\n"
        prompt = f"{header}Argument: \"{row['argument']}\"\nBelief: \"{row['topic']}\"\nAnswer:"
        return _call_api_with_retry(prompt, model, ["ATTACK", "SUPPORT"])

    def process_exp6_cot_few_shot(row):
        cot_prompt = (f"{row['final_prompt']}\n\nReason in the following way:\n1. Analyze the core assertion of the argument and the topic.\n"
                      f"2. Does the argument provide evidence, data, or reasoning that directly confirms or contradicts the topic?\n"
                      f"3. Based on this relationship, conclude if the argument supports or attacks the topic.\nConclusion:")
        example_1 = ("Argument: \"IWhat we acquired from team work is not only how to achieve the same goal with others but more importantly, how to get along with other\"\n"
                     "Belief: \"through cooperation, children can learn about interpersonal skills which are significant in the future life of all students.\"\nAnswer: SUPPORT")
        example_2 = ("Argument: \"competition makes the society more effective\"\n"
                     "Belief: \"we should attach more importance to cooperation during primary education\"\nAnswer: ATTACK")
        header = f"Here are some examples:\n\n[EXAMPLE 1]\n{example_1}\n\n[EXAMPLE 2]\n{example_2}\n\n---\n\nNow, complete the following task:\n"
        prompt = f"{header}{cot_prompt}"
        return _call_api_with_retry(prompt, model, ["ATTACK", "SUPPORT"])

    # --- List of experiments to run ---
    experiments = [
        ('gemini_without_context', process_exp1_no_context),
        ('gemini_with_context', process_exp2_with_kg),
        ('gemini_with_context_sentiment', process_exp3_with_sentiment),
        ('gemini_cot', process_exp4_cot),
        ('gemini_few_shot', process_exp5_few_shot),
        ('gemini_cot_few_shot', process_exp6_cot_few_shot)
    ]

    # --- Loop through each experiment ---
    for col_name, process_func in experiments:
        if col_name in df_exp.columns:
            print(f"ℹ️ Coluna '{col_name}' já existe. Pulando este experimento.")
            continue

        print(f"\n--- Iniciando Experimento: {col_name} ---")
        results = {}
        with ThreadPoolExecutor(max_workers=5) as executor: # Reduced workers
            futures = {executor.submit(process_func, row): index for index, row in df_exp.iterrows()}
            for future in tqdm(as_completed(futures), total=len(df_exp), desc=f"Avaliando {col_name}"):
                results[futures[future]] = future.result()

        # Add the new column to the DataFrame
        df_exp[col_name] = df_exp.index.map(results)

        # Save progress after each experiment
        output_filename = '/content/drive/My Drive/.../Mestrado/aae_evaluated_all_ways.csv'
        df_exp.to_csv(output_filename, index=False, encoding='utf-8')
        print(f"✅ Etapa concluída. Resultados parciais salvos em '{output_filename}'")

    print("\n--- Processamento de todos os experimentos concluído. ---")
    display(df_exp.head())

In [ ]:
# ==============================================================================
# CELL 4: FINAL ANALYSIS AND METRICS (Corrected)
# ==============================================================================

# Load the final evaluated file
FINAL_FILE_PATH = '/content/drive/My Drive/.../aae_evaluated_all_ways.csv'
try:
    df_final = pd.read_csv(FINAL_FILE_PATH)
    print(f"✅ Arquivo final '{FINAL_FILE_PATH}' carregado para análise.")
except FileNotFoundError:
    print(f"❌ ERRO: Arquivo final '{FINAL_FILE_PATH}' não encontrado.")
    df_final = None

def analyze_all_results(results_df):
    """Calcula e exibe métricas para todos os 6 cenários."""
    if results_df is None:
        return

    print("\n" + "="*80)
    print(" ANÁLISE DE DESEMPENHO FINAL - DATASET AAE (COMPLETO)")
    print("="*80)

    # --- FIX APPLIED HERE ---
    # Convert the ground truth column to string type to avoid mixed-type errors.
    y_true = results_df['ground_truth_stance'].astype(str)

    scenarios = {
        "1. Sem Contexto": "gemini_without_context",
        "2. Com Contexto KG": "gemini_with_context",
        "3. Com KG + Sentimento": "gemini_with_context_sentiment",
        "4. Chain-of-Thought (CoT)": "gemini_cot",
        "5. Few-Shot": "gemini_few_shot",
        "6. CoT + Few-Shot": "gemini_cot_few_shot"
    }

    labels = sorted(y_true.dropna().unique())

    for name, col in scenarios.items():
        if col in results_df.columns:
            # --- FIX APPLIED HERE ---
            # Also convert the prediction column to string type for consistency.
            y_pred = results_df[col].astype(str)

            valid_mask = y_pred.isin(labels)

            if valid_mask.sum() > 0:
                acc = accuracy_score(y_true, y_pred)
                f1 = f1_score(y_true[valid_mask], y_pred[valid_mask], average='weighted', labels=labels, zero_division=0)

                print(f"\n[Resultados para: {name}]")
                print(f"  Acurácia Geral: {acc:.4f}")
                print(f"  F1-Score (Ponderado): {f1:.4f}")

                cm = confusion_matrix(y_true[valid_mask], y_pred[valid_mask], labels=labels)
                plt.figure(figsize=(5, 3))
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
                plt.title(f'Matriz de Confusão - {name}')
                plt.ylabel('Verdadeiro')
                plt.xlabel('Predito')
                plt.show()
            else:
                print(f"\n[Resultados para: {name}] - Nenhuma resposta válida encontrada.")
        else:
            print(f"\nColuna '{col}' não encontrada para o cenário '{name}'.")

# --- Execução da Análise Final ---
if df_final is not None:
    analyze_all_results(df_final)
else:
    print("\nAnálise final não pode ser executada.")